In [1]:
from __future__ import annotations
"""
This is a playground to basically get a table to demonstrate that for multiple (small-ish) models we are able to use classifiers of either the activation space or the
input/output space to detect whether or not the model is responding about the specific subject.

NOTE in the beginning we focus on models that are 0-4B and then we will scale up to 7-10B models. After THAT we will focus on the 70-500B models.

Experiment is super simple: for each model indepednently just get all its activations for biology and then collect a very large and varied
set of activations for basically everything else. Train a logisic classifier to try and tell the difference. Here we just want to test whether
it is possible to easily seperate <something> from <everything else> for each model.

We need to be able to detect the following as being DIFFERENT from biology:
1. Prompts about subjects, such as physics, chemistry, etc...
2. Malicious prompts
3. Prompts that are simply about general world knowledge
4. Prompts that are basically not in english or are in some sense "emulating a possible latent/unwanted capability" such as ciphers
(we can play with some simple ciphers; I found a lot here: https://huggingface.co/MaxAImogs)

"""
model_names = [
    "meta-llama/Llama-3.2-1B-Instruct",
    "microsoft/Phi-4-mini-instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "google/gemma-2-2b-it",
]
from pathlib import Path
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model = model.to('cuda:3')
# pipe = pipeline(
#     "text-generation",
#     model=model_id,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )
# messages = [
#     {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
#     {"role": "user", "content": "Who are you?"},
# ]
# outputs = pipe(
#     messages,
#     max_new_tokens=256,
# )
# print(outputs[0]["generated_text"][-1])
# If memory is a concern, you can process specific layers only
camel_ai_dataset_bio = load_dataset("camel-ai/camel-ai-bio", split="train")
camel_ai_dataset_bio_tokenized = camel_ai_dataset_bio.map(lambda x: tokenizer(x["text"], padding=True, truncation=True, max_length=512), batched=True, return_tensors="pt")
def get_specific_layer_activations(text: str, layer_nums=[0, 12, 24]):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    return outputs.hidden_states#{f"layer_{i}": outputs.hidden_states[i] for i in layer_nums}
# text = "Hello, how are you?"
# activations = get_specific_layer_activations(text)
print(camel_ai_dataset_bio_tokenized)
# output_folder = Path("/mnt/align4_drive2/adrianoh/")


DatasetNotFoundError: Dataset 'camel-ai/camel-ai-bio' doesn't exist on the Hub or cannot be accessed.

In [7]:
print(type(activations))
print(len(activations))
# this appears to be some kind of residual stream...
for i in range(len(activations)):
    # print(type(activations[i]))
    print(activations[i].shape)


<class 'tuple'>
17
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
torch.Size([1, 7, 2048])
